# Exploração: fundos multimercados

Fluxo completo: localizar fundos multimercados no cadastro da CVM, baixar as cotas diárias de um período, calcular métricas e comparar com o CDI.

In [ ]:
import sys
sys.path.insert(0, "../src")

from fundos.cvm import fetch_cadastro, filtrar_multimercados, fetch_informe_periodo, cotas_do_fundo
from fundos.benchmarks import cdi_diario
from fundos.metrics import resumo, retornos_diarios, retorno_acumulado
import matplotlib.pyplot as plt

## 1. Localizar fundos multimercados

In [ ]:
cadastro = fetch_cadastro()
multimercados = filtrar_multimercados(cadastro)
multimercados[["CNPJ_FUNDO", "DENOM_SOCIAL", "CLASSE"]].head(10)

## 2. Baixar cotas de um período para um fundo escolhido

Substitua `CNPJ_ESCOLHIDO` por um CNPJ da lista acima.

In [ ]:
CNPJ_ESCOLHIDO = multimercados.iloc[0]["CNPJ_FUNDO"]

informe = fetch_informe_periodo("2026-01", "2026-07")
cotas = cotas_do_fundo(informe, CNPJ_ESCOLHIDO)
cotas.plot(title=f"Cota diária - {CNPJ_ESCOLHIDO}")

## 3. Métricas e comparação com o CDI

In [ ]:
cdi = cdi_diario(str(cotas.index.min().date()), str(cotas.index.max().date()))
print(resumo(cotas, retornos_livre_risco=cdi))

In [ ]:
retornos_fundo = retornos_diarios(cotas)
acumulado_fundo = retorno_acumulado(retornos_fundo)
acumulado_cdi = retorno_acumulado(cdi.reindex(retornos_fundo.index).dropna())

plt.plot(acumulado_fundo.index, acumulado_fundo.values, label="Fundo")
plt.plot(acumulado_cdi.index, acumulado_cdi.values, label="CDI")
plt.legend()
plt.title("Retorno acumulado: fundo vs. CDI")
plt.show()